# Publish a Measurement (V2)

This notebook is the V2 companion of `../../overview/publish_measurement.ipynb`. It publishes a waveform measurement and reads it back when NI Measurement Data Services runs in V2 mode.

The main V2 differences are:
- Data uses `ni.datastore.data.v2`. Metadata still uses `ni.datastore.metadata`, because V2 reuses the V1 Metadata API.
- Conditions belong to a step. You add them to the step instead of publishing them separately.
- Measurements are created with `create_measurement`.
- Test results and steps have a `status` (how far the work has progressed) that is separate from the `outcome` (whether it passed).

## Create a client

To get started creating metadata and publishing measurements, the client code must first instantiate a `MetadataStoreClient` and a V2 `DataStoreClient`. The `MetadataStoreClient` is used for creating and querying metadata, and the `DataStoreClient` is used for creating and querying test results, steps, and measurements.

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.data.v2 import DataStoreClient
from ni.datastore.metadata import MetadataStoreClient

metadata_store_client = MetadataStoreClient(grpc_channel=grpc_channel)
data_store_client = DataStoreClient(grpc_channel=grpc_channel)

__Recommendation:__ A client can be used within the context of a `with` statement in order to have its lifetime managed for it. Upon exiting the `with` statement, any gRPC channels owned by the client will be closed.

In [ ]:
with DataStoreClient(grpc_channel=grpc_channel) as data_store_client_one:
    query_results = data_store_client_one.query_test_results()
    # ... Perform any further operations with data_store_client_one ...

# data_store_client_one will automatically close upon exiting the 'with' block

Alternatively, the `close()` method of the client may be called to perform the same cleanup that occurs when exiting the `with` block above.

In [ ]:
data_store_client_two = DataStoreClient(grpc_channel=grpc_channel)

query_results = data_store_client_two.query_test_results()
# ... Perform any further operations with data_store_client_two ...

# Manually close the client when done using it
data_store_client_two.close()

Because notebooks such as the current example do not support splitting a single `with` statement across multiple code snippets, this example opts to call `close()` manually in the last code snippet.

## Create the test result metadata

Create the metadata that describes who ran the test, where it ran, and which software was used. Then describe the test result.

The V1 notebook also registers a metadata schema here. V2 does not accept a `schema_id` on test results, steps, or measurements yet, so this notebook stores its custom data in `extension` without a schema.

In [ ]:
from ni.datastore.data.v2 import Status, TestResult
from ni.datastore.metadata import Operator, SoftwareItem, TestStation

operator_id = metadata_store_client.create_operator(
    Operator(name="James Bowery", role="Test Operator")
)
test_station_id = metadata_store_client.create_test_station(TestStation(name="TestStation_12"))
software_item_id = metadata_store_client.create_software_item(
    SoftwareItem(product="Windows", version="10.0.19044")
)

# The test is starting, so its status is RUNNING. The outcome is set when it finishes.
test_result = TestResult(
    name="sample publish test result",
    operator_id=operator_id,
    test_station_id=test_station_id,
    software_item_ids=[software_item_id],
    status=Status.RUNNING,
    extension={"session_file_path": "C:\\my_test_description.xlsx"},
)

## Create the test result object

Now that we have the test result metadata, we can create the test result object.

In [ ]:
test_result_id = data_store_client.create_test_result(test_result)
print(f"created test result id: {test_result_id}")

## Publish the data

In V2, a step owns its conditions. Add them when you create the step, or later with `add_condition`. Then create the measurement on the step.

In [ ]:
from datetime import timezone

import hightime as ht
import numpy as np
from nitypes.waveform import AnalogWaveform, Timing

from ni.datastore.data.v2 import Condition, Outcome, Step

name = "data publish sample"
waveform = AnalogWaveform(
    sample_count=3,
    raw_data=np.array([1.0, 2.0, 3.0]),
    timing=Timing.create_with_regular_interval(
        ht.timedelta(seconds=1e-3),
        ht.datetime.now(timezone.utc)
    )
)
step = Step(
    name="Initial step",
    test_result_id=test_result_id,
    conditions=[
        Condition("Ambient Temperature", 23.0, condition_type="Environment", units="degC"),
    ],
)
step_id = data_store_client.create_step(step)
published_measurement_id = data_store_client.create_measurement(
    name=name,
    value=waveform,
    step_id=step_id,
    units="V",
    outcome=Outcome.PASSED,
)
print(f"The published measurement id is {published_measurement_id}")

# The test is finished, so mark it as done and record the outcome.
data_store_client.update_test_result(test_result_id, status=Status.DONE, outcome=Outcome.PASSED)

## Query the data

Now that we've got data in the database, we can query for it using our OData query API.

For non-scalar values, such as waveforms, `get_measurement` and `query_measurements` return the measurement details without the value. Use `read_measurement_value` to read the value.

In [ ]:
published_measurements = data_store_client.query_measurements(
    odata_query=f"$filter=Id eq {published_measurement_id}"
)
found_measurement = next(iter(published_measurements), None)
assert found_measurement is not None

test_result = data_store_client.get_test_result(found_measurement.test_result_id)
operator = metadata_store_client.get_operator(test_result.operator_id)
print(f"test result '{test_result.name}' is {test_result.status.name}: {test_result.outcome.name}")
print(f"operator: {operator.name}")
print(f"session file: {test_result.extension['session_file_path']}")

step = data_store_client.get_step(found_measurement.step_id)
for condition in step.conditions:
    print(f"condition: {condition.name} = {condition.value} {condition.units}")

waveform = data_store_client.read_measurement_value(found_measurement, expected_type=AnalogWaveform)
print(f"published data is: {waveform.raw_data}")
assert waveform.raw_data.tolist() == [1.0, 2.0, 3.0]

## Clean up

In [ ]:
metadata_store_client.close()
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()